In [ ]:
!pip -q install deep-translator


# SCALE Lab 25 Project 3 LLM: Character Transformer Baseline

Objective: train a character-level Transformer language model on the provided Lu Xun corpus and track validation perplexity.

Competition constraints from Kaggle:

- Architecture: Transformer.
- Layers: at most 6.
- Hidden dimension: at most 512.
- Attention heads: at most 8.
- Total parameters: at most 50M.
- Vocabulary size: at most 8k.
- Tokenizer: character-level tokenizer built from unique characters and punctuation in the training text.
- Metric: perplexity (PPL).


## Plan

1. Load `train.csv` and `test.csv` from the Kaggle input directory.
2. Infer the text column, then build a character vocabulary from the training text.
3. Train a compact causal Transformer baseline.
4. Report validation loss and perplexity.
5. Save model/tokenizer artifacts and generate optional test diagnostics.

The fetched Kaggle metadata did not include a sample submission file, so this notebook focuses on a reproducible model baseline and artifact generation. The final cell writes `submission.csv` only if a clear target-like column is present in `test.csv`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

KAGGLE_INPUT = Path('/kaggle/input/scale-lab-25-project-3-llm')
LOCAL_INPUT = Path('../data/scale-lab-25-project-3-llm')
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
# Choose "debug" for a fast smoke test, or "real" for a full Kaggle run.
RUN_MODE = "debug"

@dataclass
class Config:
    context_length: int = 128
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 4
    dim_feedforward: int = 1024
    dropout: float = 0.10
    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    train_fraction: float = 0.95
    max_vocab_size: int = 8000
    max_train_batches: int | None = None
    max_valid_batches: int | None = None


def make_config(run_mode: str) -> Config:
    run_mode = run_mode.lower().strip()
    if run_mode == "debug":
        return Config(
            context_length=64,
            d_model=128,
            n_heads=4,
            n_layers=2,
            dim_feedforward=512,
            batch_size=32,
            epochs=1,
            max_train_batches=20,
            max_valid_batches=10,
        )
    if run_mode == "real":
        return Config(
            context_length=128,
            d_model=256,
            n_heads=4,
            n_layers=4,
            dim_feedforward=1024,
            batch_size=64,
            epochs=3,
            max_train_batches=None,
            max_valid_batches=None,
        )
    raise ValueError(f'Unknown RUN_MODE={run_mode!r}. Use "debug" or "real".')


cfg = make_config(RUN_MODE)
print(f'RUN_MODE: {RUN_MODE}')
asdict(cfg)


## Load Data

The notebook infers the main text column by selecting the object/string column with the largest total text length. If the competition schema is different, set `TEXT_COLUMN` manually after inspecting the printed columns.


In [ ]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'Missing train.csv at {TRAIN_PATH}. On Kaggle, add the competition dataset to the notebook.')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else pd.DataFrame()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
display(train_df.head())
if not test_df.empty:
    display(test_df.head())


In [ ]:
def infer_text_column(df: pd.DataFrame) -> str:
    object_cols = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])]
    if not object_cols:
        raise ValueError('No text-like columns found. Set TEXT_COLUMN manually.')
    scores = {c: df[c].fillna('').astype(str).str.len().sum() for c in object_cols}
    return max(scores, key=scores.get)

TEXT_COLUMN = infer_text_column(train_df)
print('Using text column:', TEXT_COLUMN)

train_texts = train_df[TEXT_COLUMN].fillna('').astype(str).tolist()
corpus = '\n'.join(train_texts)
print(f'Training texts: {len(train_texts):,}')
print(f'Corpus characters: {len(corpus):,}')
print('Preview:')
print(corpus[:500])


## Reading The Lu Xun Corpus

The training file is described by the competition as `提供的鲁迅文集文本`, meaning **the provided collection of Lu Xun writings**. Lu Xun (`鲁迅`, 1881-1936) is a central figure in modern Chinese literature, known for short stories and essays written in early modern Chinese.

Because this notebook is for a Kaggle competition, translations are included only as an **exploration aid**. They are not used for tokenization, training, validation, or submission generation. The competition rules say training must use only the provided data, so external translation services should not become model inputs or labels.

The next cells print many Chinese snippets from `train.csv`. If a translation package is available, they also show rough English machine translations. These translations are intended to help a non-Chinese speaker understand the corpus, not to produce polished literary translations.


In [ ]:
def clean_for_reading(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

READING_EXAMPLES = 24
SNIPPET_CHARS = 180
rng = np.random.default_rng(SEED)

non_empty_texts = [clean_for_reading(t) for t in train_texts if clean_for_reading(t)]
if not non_empty_texts:
    raise ValueError('No non-empty training texts found to display.')

sample_indices = rng.choice(len(non_empty_texts), size=min(READING_EXAMPLES, len(non_empty_texts)), replace=False)
reading_examples = []
for n, idx in enumerate(sample_indices, start=1):
    text = non_empty_texts[int(idx)]
    if len(text) > SNIPPET_CHARS:
        start = int(rng.integers(0, max(1, len(text) - SNIPPET_CHARS)))
        snippet = text[start:start + SNIPPET_CHARS]
    else:
        start = 0
        snippet = text
    reading_examples.append({'example': n, 'source_row': int(idx), 'start_char': start, 'chinese_text': snippet})

examples_df = pd.DataFrame(reading_examples)
pd.set_option('display.max_colwidth', 260)
display(examples_df)


In [ ]:
def build_translator():
    try:
        from deep_translator import GoogleTranslator
        translator = GoogleTranslator(source='zh-CN', target='en')
        return lambda text: translator.translate(text)
    except Exception as exc:
        print('Machine translation is unavailable in this environment.')
        print('To enable it in a Kaggle notebook with internet access, run: !pip -q install deep-translator')
        print(f'Translator import/error detail: {type(exc).__name__}: {exc}')
        return None

TRANSLATE_EXAMPLES = True
translator = build_translator() if TRANSLATE_EXAMPLES else None

translated_rows = []
for row in reading_examples:
    chinese = row['chinese_text']
    if translator is None:
        english = '[translation unavailable: install deep-translator or use another translation service for exploration only]'
    else:
        try:
            english = translator(chinese)
        except Exception as exc:
            english = f'[translation failed: {type(exc).__name__}: {exc}]'
    translated_rows.append({**row, 'english_machine_translation': english})

translated_df = pd.DataFrame(translated_rows)
display(translated_df[['example', 'chinese_text', 'english_machine_translation']])


## Character Tokenizer

The tokenizer is deliberately simple: one Unicode character per token. It is fitted only on the training corpus, with `<pad>` and `<unk>` reserved for batching and unseen characters.


In [ ]:
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN]

chars = sorted(set(corpus))
max_chars = cfg.max_vocab_size - len(SPECIAL_TOKENS)
if len(chars) > max_chars:
    raise ValueError(f'Character vocabulary has {len(chars)} unique chars, exceeding limit {max_chars}.')

itos = SPECIAL_TOKENS + chars
stoi = {ch: i for i, ch in enumerate(itos)}
pad_id = stoi[PAD_TOKEN]
unk_id = stoi[UNK_TOKEN]

print(f'Vocabulary size: {len(itos):,}')
print('First 100 vocab entries:', itos[:100])
assert len(itos) <= cfg.max_vocab_size

def encode(text: str) -> list[int]:
    return [stoi.get(ch, unk_id) for ch in text]

def decode(ids: list[int]) -> str:
    return ''.join(itos[i] for i in ids if i >= len(SPECIAL_TOKENS))

encoded = np.array(encode(corpus), dtype=np.int64)
print('Encoded length:', len(encoded))


In [ ]:
split_idx = int(len(encoded) * cfg.train_fraction)
train_ids = encoded[:split_idx]
valid_ids = encoded[max(0, split_idx - cfg.context_length):]

print(f'Train tokens: {len(train_ids):,}')
print(f'Valid tokens: {len(valid_ids):,}')

class CharBlockDataset(Dataset):
    def __init__(self, token_ids: np.ndarray, context_length: int):
        if len(token_ids) <= context_length + 1:
            raise ValueError('Not enough tokens for the configured context length.')
        self.token_ids = token_ids
        self.context_length = context_length

    def __len__(self) -> int:
        return len(self.token_ids) - self.context_length - 1

    def __getitem__(self, idx: int):
        chunk = self.token_ids[idx: idx + self.context_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

train_ds = CharBlockDataset(train_ids, cfg.context_length)
valid_ds = CharBlockDataset(valid_ids, cfg.context_length)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
valid_loader = DataLoader(valid_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, y_batch.shape)
print(decode(x_batch[0].tolist()[:120]))


## Model

This is a causal Transformer encoder used as a decoder-only language model. The causal mask prevents positions from attending to future characters.


In [ ]:
class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, config: Config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.n_heads,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=config.n_layers)
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=input_ids.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=causal_mask)
        x = self.norm(x)
        return self.lm_head(x)

model = CausalTransformerLM(len(itos), cfg).to(device)
param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {param_count:,}')
print(f'Trainable parameters: {trainable_count:,}')
assert cfg.n_layers <= 6
assert cfg.d_model <= 512
assert cfg.n_heads <= 8
assert param_count <= 50_000_000


In [ ]:
def run_epoch(loader: DataLoader, train: bool) -> float:
    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    max_batches = cfg.max_train_batches if train else cfg.max_valid_batches

    for step, (x, y) in enumerate(loader, start=1):
        if max_batches is not None and step > max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            scheduler.step()

        token_count = y.numel()
        total_loss += loss.item() * token_count
        total_tokens += token_count

    return total_loss / max(1, total_tokens)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
total_steps = len(train_loader) * cfg.epochs if cfg.max_train_batches is None else cfg.max_train_batches * cfg.epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps))

history = []
best_valid_loss = float('inf')
best_model_path = OUTPUT_DIR / 'char_transformer_baseline.pt'

for epoch in range(1, cfg.epochs + 1):
    start = time.time()
    train_loss = run_epoch(train_loader, train=True)
    valid_loss = run_epoch(valid_loader, train=False)
    valid_ppl = math.exp(min(20, valid_loss))
    elapsed = time.time() - start

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'valid_loss': valid_loss,
        'valid_ppl': valid_ppl,
        'seconds': elapsed,
    }
    history.append(row)
    print(row)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
display(history_df)
print('Best validation PPL:', math.exp(min(20, best_valid_loss)))
print('Saved model:', best_model_path)


## Qualitative Sampling

Sampling is not used for scoring perplexity, but it is useful for catching obvious training failures.


In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 300, temperature: float = 0.9, top_k: int = 50) -> str:
    model.eval()
    ids = encode(prompt)
    if not ids:
        ids = [unk_id]
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context = input_ids[:, -cfg.context_length:]
        logits = model(context)[:, -1, :] / max(temperature, 1e-6)
        logits[:, pad_id] = -float('inf')
        if top_k is not None and top_k > 0:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            threshold = values[:, [-1]]
            logits = torch.where(logits < threshold, torch.full_like(logits, -float('inf')), logits)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return decode(input_ids[0].tolist())

prompt = corpus[: min(80, len(corpus))]
print(generate(prompt, max_new_tokens=300))


## Save Artifacts

Artifacts include tokenizer mappings, model configuration, training history, and the best model weights.


In [ ]:
tokenizer_path = OUTPUT_DIR / 'char_tokenizer.json'
config_path = OUTPUT_DIR / 'char_transformer_config.json'
history_path = OUTPUT_DIR / 'training_history.csv'

with tokenizer_path.open('w', encoding='utf-8') as f:
    json.dump({'itos': itos, 'pad_token': PAD_TOKEN, 'unk_token': UNK_TOKEN, 'text_column': TEXT_COLUMN}, f, ensure_ascii=False, indent=2)

with config_path.open('w', encoding='utf-8') as f:
    json.dump({**asdict(cfg), 'vocab_size': len(itos), 'param_count': param_count}, f, ensure_ascii=False, indent=2)

history_df.to_csv(history_path, index=False)
print('Saved:', tokenizer_path)
print('Saved:', config_path)
print('Saved:', history_path)


## Submission File And Model Diagnostics

Kaggle has now given two useful schema errors:

- The submission must contain an `id` column.
- The submission must contain **5 rows**, not one row for every row in `test.csv`.

Because `test.csv` appears to contain 100 rows with `name`, `type`, and `context`, this notebook treats those rows as candidate contexts. It runs each candidate `context` through the trained Transformer, computes row-level negative log-likelihood/perplexity, and selects the lowest-perplexity candidate from each prompt group.

The notebook tries to infer the 5 prompt groups in this order:

1. unique `(name, type)` pairs, if there are 5;
2. unique `name` values, if there are 5;
3. unique `type` values, if there are 5;
4. otherwise, the 5 lowest-perplexity rows overall as a fallback.

The official `submission.csv` contains only `id`, `name`, `type`, and `context`. Full model scores are saved separately in `test_diagnostics.csv`.


In [ ]:
submission_path = OUTPUT_DIR / 'submission.csv'
diagnostics_path = OUTPUT_DIR / 'test_diagnostics.csv'

if test_df.empty:
    raise ValueError('No test.csv found; cannot create submission.csv.')

required_test_columns = ['name', 'type', 'context']
missing_columns = [col for col in required_test_columns if col not in test_df.columns]
if missing_columns:
    raise ValueError(f'test.csv is missing expected columns: {missing_columns}. Found: {test_df.columns.tolist()}')

sample_submission_paths = sorted(DATA_DIR.glob('*sample*submission*.csv'))
sample_submission = pd.read_csv(sample_submission_paths[0]) if sample_submission_paths else None
expected_rows = len(sample_submission) if sample_submission is not None else 5
print('Expected submission rows:', expected_rows)
if sample_submission is not None:
    print('Found sample submission:', sample_submission_paths[0])
    print('Sample submission columns:', sample_submission.columns.tolist())

if best_model_path.exists():
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print('Loaded best model weights:', best_model_path)

model.eval()
test_texts = test_df['context'].fillna('').astype(str).tolist()

@torch.no_grad()
def text_nll_and_ppl(text: str) -> tuple[float, float, int]:
    ids = encode(text)
    if len(ids) <= 1:
        return float('nan'), float('nan'), len(ids)

    total_loss = 0.0
    total_tokens = 0
    for start in range(0, len(ids) - 1, cfg.context_length):
        chunk = ids[start: start + cfg.context_length + 1]
        if len(chunk) <= 1:
            continue
        x = torch.tensor(chunk[:-1], dtype=torch.long, device=device).unsqueeze(0)
        y = torch.tensor(chunk[1:], dtype=torch.long, device=device).unsqueeze(0)
        logits = model(x)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            reduction='sum',
        )
        total_loss += float(loss.item())
        total_tokens += int(y.numel())

    if total_tokens == 0:
        return float('nan'), float('nan'), len(ids)
    mean_nll = total_loss / total_tokens
    ppl = math.exp(min(20, mean_nll))
    return mean_nll, ppl, len(ids)

rows = []
for row_idx, text in enumerate(test_texts):
    nll, ppl, token_count = text_nll_and_ppl(text)
    rows.append({
        'source_row': row_idx,
        'source_id': test_df['id'].iloc[row_idx] if 'id' in test_df.columns else row_idx,
        'name': test_df['name'].iloc[row_idx],
        'type': test_df['type'].iloc[row_idx],
        'context': test_df['context'].iloc[row_idx],
        'nll': nll,
        'ppl': ppl,
        'token_count': token_count,
    })

diagnostics = pd.DataFrame(rows)

def choose_group_keys(df: pd.DataFrame, expected_n: int) -> list[str] | None:
    candidates = [
        ['name', 'type'],
        ['name'],
        ['type'],
    ]
    for keys in candidates:
        if df[keys].drop_duplicates().shape[0] == expected_n:
            return keys
    return None

group_keys = choose_group_keys(diagnostics, expected_rows)
if group_keys is None:
    print('Could not infer 5 prompt groups; using the lowest-perplexity rows overall.')
    selected = diagnostics.nsmallest(expected_rows, 'ppl').copy()
else:
    print('Selecting lowest-perplexity context per group:', group_keys)
    selected_idx = diagnostics.groupby(group_keys, dropna=False)['ppl'].idxmin()
    selected = diagnostics.loc[selected_idx].sort_values(group_keys).copy()

selected = selected.reset_index(drop=True)
if len(selected) != expected_rows:
    raise ValueError(f'Built {len(selected)} submission rows, but expected {expected_rows}. Check grouping logic.')

submission = selected[['name', 'type', 'context']].copy()
submission.insert(0, 'id', np.arange(len(submission)))

# If Kaggle exposes a sample submission with a different column order, follow it where possible.
if sample_submission is not None:
    for col in sample_submission.columns:
        if col not in submission.columns:
            submission[col] = sample_submission[col].values[:len(submission)]
    submission = submission[[col for col in sample_submission.columns if col in submission.columns]]

submission.to_csv(submission_path, index=False)
diagnostics.to_csv(diagnostics_path, index=False)

print('Saved submission:', submission_path)
print('Saved model diagnostics:', diagnostics_path)
print('Submission shape:', submission.shape)
print('Submission columns:', submission.columns.tolist())
display(submission)
display(diagnostics[['source_row', 'name', 'type', 'nll', 'ppl', 'token_count']].head())


## Notes

- Increase `epochs` for a real run after confirming the notebook executes end-to-end.
- Try `d_model=384`, `n_layers=6`, and `n_heads=6` or `8` if Kaggle runtime allows and the parameter count remains below 50M.
- Longer `context_length` can improve language modeling but increases memory use quadratically in attention.
